In [1]:
%run ../stage1.py \
    --sc_file "/mnt/d/ST_Graduation_Project_data/DATA24/scRNA.h5ad" \
    --st_file "/mnt/d/ST_Graduation_Project_data/DATA24/Spatial.h5ad" \
    --n_epochs 150 \
    --resolution 4 \
    --top_n_per_type 100 \
    --output_dir ../deconv_results/DATA24 \
    --marker_selection_method l1 \
    --precomputed_marker_file "../deconv_results/DATA24_backup/final_genes.txt"

/home/mwc/miniconda3/envs/dl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Stage 1 Training: VAE (SC + ST, Marker Genes)
Configuration:
   Marker genes per type: 100
   Leiden resolution: 4.0
   Precomputed marker genes: ../deconv_results/DATA24_backup/final_genes.txt
   Batch size: 256
   Epochs: 150
   Learning rate: 0.0005
   Beta (KL weight): 0.1
   Hidden dims: [512, 256]
   Latent dim: 128
   Loss type: MSE
   Lambda MMD: 0
   Dual Decoder: True
   Aggregation method: mean
Loading datasets...
   Loading SC: /mnt/d/ST_Graduation_Project_data/DATA24/scRNA.h5ad
   SC shape: (10000, 31489)
   SC avg counts/cell: 933.9 (from X)
   Loading ST: /mnt/d/ST_Graduation_Project_data/DATA24/Spatial.h5ad
   ST shape: (1000, 27345)
   Common genes: 17733
   SC final: (10000, 17733)
   ST final: (1000, 17733)
Using precomputed marker genes from: ../deconv_results/DATA24_backup/final_genes.txt
Loading marker genes from file: ../deconv_results/DATA24_backup/final_genes.txt
   Loaded 824 marker genes
Saved clustered SC adata: ../deconv_results/DATA24/sc_

VAE Training: 100%|██████████| 150/150 [01:51<00:00,  1.35epoch/s, Train=652.6056, Recon=646.0458, KL=65.5980, Test=680.0020]


📊 Computing cluster centers using ALL SC data (train + test)...
   ALL SC data: (10000, 824)
   Number of clusters: 53
   Computed embeddings shape: (10000, 128)
Computing cluster centers with mean aggregation...
   Completed: 53 clusters with mean centers and expressions
Extracting celltype-cluster mapping (using 'cell_type' column)...
Visualizing modality alignment...
Generating UMAP visualization for modality alignment...
   Computing UMAP on 10890 samples with 128 dims...
   UMAP visualization saved to: ../deconv_results/DATA24/modality_alignment_umap.png
Saving model to: ../deconv_results/DATA24/final_vae.pth
   Average cell counts: 933.9 (for Stage 2 scale factor)
✅ Saved VAE model (weights only): ../deconv_results/DATA24/final_vae.pth
Saving cluster data to: ../deconv_results/DATA24/final_vae_cluster_data.npz
   ✓ Cluster IDs: 53
   ✓ Prototypes: (53, 128)
   ✓ Expressions (marker): (53, 824)
   ✓ Expressions (full): 53 clusters × 17733 genes
   ✓ Celltype mapping: 53 clusters
✅

In [2]:
%run ../stage2.py \
    --st_file "/mnt/d/ST_Graduation_Project_data/DATA24/Spatial.h5ad" \
    --stage1_model_path "../deconv_results/DATA24/final_vae.pth" \
    --output_dir "../deconv_results/DATA24/" \
    --gat_hidden_dim 512 \
    --gat_layers 4 \
    --lr 5e-3 \
    --k_spatial 5 \
    --k_celltype 20 \
    --batch_size 512 \
    --n_epochs 200

Sample name: Spatial
Stage 1 model: ../deconv_results/DATA24/final_vae.pth
Output directory: ../deconv_results/DATA24/
Device: cpu
Weight threshold: 0.0001
Loading pretrained VAE Encoder...
   VAE architecture: 824 -> 128
   Output type: mse
   Architecture: Dual Decoder (SC/ST-specific)
   ✓ Loaded 10000 cell cluster labels from checkpoint
Loading cluster data from: ../deconv_results/DATA24/final_vae_cluster_data.npz
   Cluster prototypes: torch.Size([53, 128])
   Cluster expressions (marker): torch.Size([53, 824])
   Cluster expressions (all genes): 53 × 17733
   Loaded celltype mapping: 53 clusters
   Average cell counts: 933.9
Loaded all genes list: 17733 genes
VAE Encoder loaded: 824 -> 128
Cell type clusters: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '5', '50'

GAT Training: 100%|██████████| 200/200 [06:43<00:00,  2.02s/epoch, Total=3.3403, MSE=0.2290, Spot_Cosine=0.1667, Gene_Cosine=0.3456, Pearson=0.2559, Gene_Pearson=0.5620, P_pat=7, M_pat=3, C_pat=7]   


Evaluating model results...
Applying weight threshold: 0.0001
   Non-zero elements: 20000 -> 14651 (27.6%)
Saving deconvolution results...
Generating deconvolution expression matrices...
   Reconstructing all gene expression...
   Saved: ../deconv_results/DATA24//Spatial_reconstructed_all_genes.csv
   Cell type composition...
   Found duplicate celltype names: ['Proximal Tubule', 'Endothelial Cell', 'Connecting Tubule', 'Thick Ascending Limb', 'Principal Cells', 'Distal Convoluted Tubule', 'Intercalated Cells Type A']. Merging corresponding cluster columns by summing weights.
   Columns before: 53, after merge: 10
   Saved cell composition (celltype): ../deconv_results/DATA24//Spatial_cell_composition.csv
   Saved cluster composition: ../deconv_results/DATA24//Spatial_cluster_composition.csv
   Using 824 marker genes for reconstruction quality (consistent with training objective)
   Cosine similarities saved: ../deconv_results/DATA24//Spatial_spot_cosine_similarity.csv

Plotting recons